# Modeling - Ferry Ticket Demand Forecasting

This notebook covers model training, evaluation, and comparison for ferry ticket demand forecasting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from preprocess import FerryDataPreprocessor, create_sample_data
from features import FerryFeatureEngineer
from train import setup_default_models, ModelTrainer
from evaluate import ModelEvaluator
from forecast import FerryForecaster, ForecastPipeline

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

## 1. Load and Prepare Data

In [ ]:
# Create sample data if not exists
sample_path = "../data/raw/ferry_sample_data.csv"
if not Path(sample_path).exists():
    create_sample_data(sample_path, days=30)
    print("Sample data created")

# Load and preprocess
preprocessor = FerryDataPreprocessor()
df = preprocessor.load_data(sample_path)
df = preprocessor.preprocess_pipeline(df)

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

In [ ]:
# Feature engineering
engineer = FerryFeatureEngineer()
df_features = engineer.create_all_features(df)

print(f"Feature matrix shape: {df_features.shape}")
print(f"Number of features: {len(engineer.feature_names)}")

## 2. Prepare Training Data for Different Horizons

In [ ]:
# Get feature matrices for all horizons
horizons = ['15m', '30m', '1h', '2h']
feature_matrices = {}

for horizon in horizons:
    X, y = engineer.get_feature_matrix(df_features, horizon=horizon)
    feature_matrices[horizon] = (X, y)
    print(f"{horizon} horizon: X shape = {X.shape}, y shape = {y.shape}")

## 3. Train Models for 1-Hour Horizon

In [ ]:
# Focus on 1-hour horizon for detailed modeling
X, y = feature_matrices['1h']

# Time-based split
trainer = setup_default_models()
X_train, X_test, y_train, y_test = trainer.time_series_split(X, y, test_size=0.2)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining date range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Test date range: {X_test.index.min()} to {X_test.index.max()}")

In [ ]:
# Train all models
print("Training models...")
trained_models = trainer.train_all_models(X_train, y_train)
print(f"\nSuccessfully trained {len(trained_models)} models")

## 4. Evaluate Models

In [ ]:
# Evaluate all models
evaluator = ModelEvaluator()

for model_name, model in trained_models.items():
    try:
        y_pred = model.predict(X_test)
        metrics = evaluator.evaluate_model(model_name, y_test.values, y_pred)
        print(f"{model_name}: MAE = {metrics['MAE']:.4f}, RMSE = {metrics['RMSE']:.4f}")
    except Exception as e:
        print(f"Error evaluating {model_name}: {e}")

In [ ]:
# Model comparison table
comparison_df = evaluator.compare_models()
comparison_df

In [ ]:
# Get best model by MAE
best_model, best_mae = evaluator.get_best_model('MAE')
print(f"Best model by MAE: {best_model} ({best_mae:.4f})")

# Get best model by RMSE
best_model_rmse, best_rmse = evaluator.get_best_model('RMSE')
print(f"Best model by RMSE: {best_model_rmse} ({best_rmse:.4f})")

## 5. Visualize Model Performance

In [ ]:
# Plot model comparison
fig_comparison = evaluator.plot_model_comparison('MAE')
fig_comparison.show()

In [ ]:
# Plot forecast vs actual for best model
fig_forecast = evaluator.plot_forecast_vs_actual(best_model, X_test.index, n_points=100)
fig_forecast.show()

In [ ]:
# Plot forecast with confidence intervals
fig_ci = evaluator.plot_forecast_with_confidence(best_model, X_test.index, confidence=0.95, n_points=100)
fig_ci.show()

In [ ]:
# Plot error distribution
fig_error = evaluator.plot_error_distribution(best_model)
fig_error.show()

## 6. Train Models for All Horizons

In [ ]:
# Train and evaluate for all horizons
horizon_evaluators = {}
horizon_models = {}

for horizon in horizons:
    print(f"\n{'='*50}")
    print(f"Training models for {horizon} horizon")
    print(f"{'='*50}")
    
    X, y = feature_matrices[horizon]
    
    # Time-based split
    trainer_h = setup_default_models()
    X_train_h, X_test_h, y_train_h, y_test_h = trainer_h.time_series_split(X, y, test_size=0.2)
    
    # Train models
    trained_models_h = trainer_h.train_all_models(X_train_h, y_train_h)
    horizon_models[horizon] = trained_models_h
    
    # Evaluate
    evaluator_h = ModelEvaluator()
    for model_name, model in trained_models_h.items():
        try:
            y_pred_h = model.predict(X_test_h)
            evaluator_h.evaluate_model(model_name, y_test_h.values, y_pred_h)
        except Exception as e:
            print(f"Error evaluating {model_name}: {e}")
    
    horizon_evaluators[horizon] = evaluator_h
    
    # Print comparison
    comparison_h = evaluator_h.compare_models()
    print(f"\n{horizon} Horizon Model Comparison:")
    print(comparison_h[['MAE', 'RMSE', 'MAPE']])

## 7. Horizon-Wise Analysis

In [ ]:
# Calculate horizon-wise metrics
from evaluate import calculate_horizon_wise_metrics

horizon_metrics_df = calculate_horizon_wise_metrics(horizon_evaluators)
horizon_metrics_df

In [ ]:
# Plot MAE across horizons for each model
fig = go.Figure()

for model in horizon_metrics_df['model'].unique():
    model_data = horizon_metrics_df[horizon_metrics_df['model'] == model]
    fig.add_trace(go.Scatter(
        x=model_data['horizon'],
        y=model_data['MAE'],
        mode='lines+markers',
        name=model
    ))

fig.update_layout(
    title='MAE Across Forecast Horizons',
    xaxis_title='Forecast Horizon',
    yaxis_title='MAE',
    template='plotly_white',
    height=500
)

fig.show()

## 8. Save Trained Models

In [ ]:
# Save models for 1-hour horizon (primary use case)
trainer.save_all_models("../models")
print("Models saved successfully")

## 9. Generate Evaluation Report

In [ ]:
# Generate comprehensive evaluation report
evaluator.generate_evaluation_report("../reports/model_evaluation_report.txt")
print("Evaluation report generated")

## 10. Test Forecasting Pipeline

In [ ]:
# Initialize forecasting pipeline
forecaster = FerryForecaster("../models")

# Load models
forecaster.load_all_models()
print(f"Loaded {len(forecaster.models)} models")

In [ ]:
# Generate forecast for next time steps
X_forecast = X_test.tail(10)
timestamps_forecast = X_forecast.index

# Generate forecast report
thresholds = {'critical': 100, 'high': 75, 'medium': 50, 'low': 0}
forecast_report = forecaster.generate_forecast_report(
    X_forecast,
    timestamps_forecast,
    model_name='Random Forest',
    confidence=0.95,
    thresholds=thresholds
)

print("Sample Forecast Report:")
forecast_report.head()

## 11. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest
if 'rf' in trained_models:
    rf_model = trained_models['rf']
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_model.model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('importance', ascending=False)
    
    print("Top 20 Most Important Features:")
    print(feature_importance.head(20))
    
    # Plot feature importance
    fig = go.Figure(data=[go.Bar(
        x=feature_importance['importance'].head(20),
        y=feature_importance['feature'].head(20),
        orientation='h',
        marker_color='steelblue'
    )])
    
    fig.update_layout(
        title='Top 20 Feature Importance (Random Forest)',
        xaxis_title='Importance',
        yaxis_title='Feature',
        template='plotly_white',
        height=600
    )
    
    fig.show()

## 12. Summary and Recommendations

In [ ]:
print("=== MODELING SUMMARY ===")
print(f"\nTotal models trained: {len(trained_models)}")
print(f"\nBest model (MAE): {best_model} ({best_mae:.4f})")
print(f"Best model (RMSE): {best_model_rmse} ({best_rmse:.4f})")

print("\n=== HORIZON PERFORMANCE ===")
for horizon, evaluator_h in horizon_evaluators.items():
    best_h, best_mae_h = evaluator_h.get_best_model('MAE')
    print(f"{horizon}: Best = {best_h} (MAE = {best_mae_h:.4f})")

print("\n=== RECOMMENDATIONS ===")
print("1. Use Random Forest for production forecasting")
print("2. Monitor model performance weekly")
print("3. Retrain models monthly with new data")
print("4. Consider ensemble methods for improved accuracy")
print("5. Implement confidence intervals for uncertainty quantification")